## Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, trim, when, row_number
from pyspark.sql.functions import sum as spark_sum_val, countDistinct, collect_set, max as spark_max, round as spark_round, current_timestamp
from pyspark.sql import Window

### Reviews Table Data Manipulation and Cleaning

In [0]:
df_reviews_bronze = spark.table("olist_ecommerce_project.bronze.brz_reviews")

# Basic profiling
print("Total rows:", df_reviews_bronze.count())
print("Distinct review_id:", df_reviews_bronze.select("review_id").distinct().count())
print("Distinct order_id:", df_reviews_bronze.select("order_id").distinct().count())

# Null check
df_reviews_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_reviews_bronze.columns
]).show()

# Review score distribution
print("\nReview score distribution:")
df_reviews_bronze.groupBy("review_score").count().orderBy("review_score").show(truncate=False)

# Check for duplicate review_id
print("Duplicate review_ids:")
df_reviews_bronze.groupBy("review_id").count().filter(col("count") > 1).count()

Checking the review_ids column to know why there are duplicates

In [0]:
# Inspect duplicate review_ids — are they exact duplicates or different data?

df_dupes = (
    df_reviews_bronze
    .groupBy("review_id")
    .count()
    .filter(col("count") > 1)
)

# Join back to see actual duplicate rows
df_dupe_details = df_reviews_bronze.join(
    df_dupes.select("review_id"),
    on="review_id",
    how="inner"
).orderBy("review_id")

print("Total rows involved in duplicates:", df_dupe_details.count())

# Are they exact duplicates (same everything) or partial (same review_id, different data)?
print("\nSample duplicate review rows:")
df_dupe_details.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message",
    "review_creation_date"
).show(10, truncate=False)

# Check if same review_id always maps to same order_id
print("\nReview_ids with multiple different order_ids:")
df_reviews_bronze.groupBy("review_id", "order_id").count()\
    .groupBy("review_id").count()\
    .filter(col("count") > 1).count()

Checking the orders table has also duplicates or not

In [0]:
# Check if any order_id has multiple different reviews
print("Orders with multiple reviews:")
df_reviews_bronze.groupBy("order_id")\
    .count()\
    .filter(col("count") > 1)\
    .count()

# What do those look like?
df_reviews_bronze.groupBy("order_id")\
    .count()\
    .filter(col("count") > 1)\
    .orderBy("count", ascending=False)\
    .show(10, truncate=False)

Checking how many orders are affected

In [0]:
# Total orders with multiple reviews
print("Orders with multiple reviews:")
multiple_reviews = df_reviews_bronze.groupBy("order_id")\
    .count()\
    .filter(col("count") > 1)
print(multiple_reviews.count())

# Peek at what multiple reviews for same order look like
df_reviews_bronze.join(
    multiple_reviews.select("order_id"),
    on="order_id",
    how="inner"
).orderBy("order_id", "review_answer_timestamp")\
.select(
    "order_id",
    "review_id",
    "review_score",
    "review_answer_timestamp"
).show(10, truncate=False)

### Creating the Silver Table

In [0]:

# Step 1: Deduplicate — keep latest review per order_id
window_latest = Window.partitionBy("order_id").orderBy(
    col("review_answer_timestamp").desc()
)

df_reviews_deduped = (
    df_reviews_bronze
    .withColumn("row_num", row_number().over(window_latest))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows after deduplication:", df_reviews_deduped.count())

# Step 2: Fill null comment columns with empty string
# Easier to handle downstream than nulls in text fields
df_reviews_silver = df_reviews_deduped.withColumn(
    "review_comment_title",
    when(col("review_comment_title").isNull(), "")
    .otherwise(trim(col("review_comment_title")))
).withColumn(
    "review_comment_message",
    when(col("review_comment_message").isNull(), "")
    .otherwise(trim(col("review_comment_message")))
)

# Step 3: Add has_comment flag
# Useful in Gold to filter reviews with actual text vs score-only
df_reviews_silver = df_reviews_silver.withColumn(
    "has_comment",
    when(col("review_comment_message") != "", True)
    .otherwise(False)
)

# Step 4: Add sentiment_category based on review_score
# Standard NPS-style bucketing used in ecommerce analytics
df_reviews_silver = df_reviews_silver.withColumn(
    "sentiment_category",
    when(col("review_score") >= 4, "positive")
    .when(col("review_score") == 3, "neutral")
    .otherwise("negative")
)

# Step 5: Drop source_file audit column
df_reviews_silver = df_reviews_silver.drop("_source_file")

# Sanity check
print("Final silver reviews rows:", df_reviews_silver.count())
df_reviews_silver.select(
    "review_id",
    "order_id",
    "review_score",
    "sentiment_category",
    "has_comment",
    "review_comment_message"
).show(10, truncate=False)

In [0]:
(
    df_reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_reviews")
)

print("slv_reviews written successfully")

## Reviews — Silver Layer Cleaning Notes

The Reviews table captures customer feedback for each order. Most customers 
only leave a star rating without any written comment, so high null rates in 
comment columns are expected and not a data quality issue. This is the last 
table in the Silver layer.

### Profiling Summary

| Metric | Value |
|---|---|
| Total rows | 99,224 |
| Distinct review_ids | 98,410 |
| Distinct order_ids | 98,673 |
| Null review_comment_title | 87,656 (~88%) |
| Null review_comment_message | 58,247 (~59%) |
| Duplicate review_ids | 789 |
| Orders with multiple reviews | 547 |

### Review Score Distribution

| Score | Count | Sentiment |
|---|---|---|
| 1 ⭐ | 11,424 | Negative |
| 2 ⭐ | 3,151 | Negative |
| 3 ⭐ | 8,179 | Neutral |
| 4 ⭐ | 19,142 | Positive |
| 5 ⭐ | 57,328 | Positive |

Heavily skewed towards 5 stars (~58%) — typical ecommerce pattern where 
customers are most motivated to review when very satisfied or very dissatisfied.

### Checks Performed

**1. Duplicate Review IDs**
- Found 789 duplicate review_ids across 99,224 rows
- Investigation revealed every duplicate review_id mapped to a 
  different order_id but had identical score, message and creation date
- Root cause: Olist platform systematically copy-pasted the same 
  review across multiple orders — a source system issue, not a 
  customer behavior pattern

**2. Orders with Multiple Reviews**
- Found 547 orders with more than one review (some with up to 3)
- Two patterns identified:
  - Pure duplicates — same score and message, different review_id
  - Genuine updates — customer submitted a new review with a 
    different score after their initial review
- Decision: Keep the latest review per order by `review_answer_timestamp`
  — handles both cases correctly. For pure duplicates it picks either 
  one (same data). For updates it picks the most recent customer sentiment.

**3. Null Comment Columns**
- `review_comment_title` → 87,656 nulls (~88%) — expected
- `review_comment_message` → 58,247 nulls (~59%) — expected
- Decision: Fill nulls with empty string rather than dropping rows
  since the star rating itself is still valuable even without a comment

**4. Referential integrity**
- All order_ids in Reviews were verified against slv_orders
- No orphaned reviews found ✅

### Transformations Applied

**1. Deduplicated by order_id**
- Kept latest review per order using row_number() over 
  review_answer_timestamp descending
- Reduced from 99,224 → 98,673 rows (one per order)

**2. Filled null comment columns**
- Replaced nulls with empty string in both comment columns
- Applied trim() to remove any leading/trailing whitespace

**3. Added `has_comment` flag**
- True if review_comment_message is not empty
- Allows Gold layer to easily filter text reviews vs score-only reviews

**4. Added `sentiment_category`**
- Bucketed review_score into 3 categories using NPS-style logic:
  - score >= 4 → positive
  - score == 3 → neutral  
  - score <= 2 → negative
- Avoids repeating this bucketing logic in every Gold query

**5. Dropped `_source_file`**
- Removed Bronze-specific audit column as per Silver layer standard

### Gold Layer Use Cases

| Column | Gold Use Case |
|---|---|
| `review_score` | Average score per seller, product category, state |
| `sentiment_category` | Positive/neutral/negative rate per seller or category |
| `has_comment` | Filter for text review analysis |
| `review_comment_message` | Qualitative feedback — future NLP/sentiment analysis |
| `review_creation_date` | Review volume trends over time |